In [ ]:
# data loading, alignment, and sequence construction for lstm

from __future__ import annotations

from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


def load_xy_pair(
    x_path: Path,
    y_path: Path,
) -> Tuple[pd.DataFrame, np.ndarray, pd.Series]:
    # load X/y CSVs, align on Date, return feature matrix, labels, and dates for sequences
    X = pd.read_csv(x_path, parse_dates=["Date"])
    y_df = pd.read_csv(y_path, parse_dates=["Date"])    

    merged = X.merge(y_df, on="Date", how="inner", validate="one_to_one")
    merged = merged.sort_values("Date").reset_index(drop=True)

    # Drop rows with missing values in features or target
    target_col = "Target_Next_Day_Direction"
    feature_cols = [c for c in X.columns if c != "Date"]
    merged = merged.dropna(subset=feature_cols + [target_col])

    y = merged[target_col].astype(np.float32).to_numpy()
    dates = merged["Date"]
    features = merged[feature_cols].astype(np.float32)
    return features, y, dates


def make_sequences(
    X: np.ndarray,
    y: np.ndarray,
    seq_len: int,
) -> Tuple[np.ndarray, np.ndarray]:
    # create 3D inputs (batch, time, features) and 1D binary targets
    # for each index i >= seq_len - 1, one sample is X[i-seq_len+1 : i+1] -> y[i]
    n = X.shape[0]
    # check if num observations is enough to make a sequence
    if n < seq_len:
        raise ValueError(f"Need at least seq_len={seq_len} rows; got {n}")

    X_seq = np.stack([X[i - seq_len + 1 : i + 1] for i in range(seq_len - 1, n)], axis=0)
    y_seq = y[seq_len - 1 :].copy()
    return X_seq.astype(np.float32), y_seq.astype(np.float32)


class SequenceDataset(Dataset):
    # PyTorch Dataset wrapping pre-built sequences

    def __init__(self, X_seq: np.ndarray, y_seq: np.ndarray):
        self.X = torch.from_numpy(X_seq)
        self.y = torch.from_numpy(y_seq)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


def time_series_chunk_indices(n_samples: int, n_chunks: int) -> list[tuple[int, int]]:
    # split [0, n_samples) into contiguous chunks for time-series CV on sequence data
    # returns list of (start, end) end-exclusive index ranges in terms of sequence samples
    if n_chunks < 2:
        raise ValueError("n_chunks must be >= 2")
    edges = np.linspace(0, n_samples, n_chunks + 1, dtype=int)
    return [(int(edges[i]), int(edges[i + 1])) for i in range(n_chunks)]
